# Sjednocení a technické čištění dat

Cílem notebooku je sjednotit výstupy jednotlivých parserů do jedné konzistentní datové sady. V této fázi se neprovádí jazykové předzpracování textu pro výpočet podobnosti, například tokenizace, lemmatizace, odstranění stop slov nebo vektorizace. Tyto kroky budou řešeny až v metodice porovnávání sylabů.

Notebook se zaměřuje na kontrolu schématu, sjednocení prázdných hodnot, technické čištění textových polí, kontrolu formálních atributů a export výsledných CSV souborů do složky `data/processed`.

In [1]:
# standardní knihovny Pythonu
import sys
import re
from pathlib import Path

# knihovny třetích stran
import pandas as pd
import numpy as np

# nastavení root složky projektu pro import utility modulů
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

# sdílené schéma datasetu
from utils.schema import FINAL_COLUMNS

# nastavení cest ve struktuře projektu
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# vytvoření výstupní složky, pokud neexistuje
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# nastavení zobrazení dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## Načtení parsed datasetů a kontrola schématu

In [2]:
# seznam parsed datasetů
DATASETS = {
    "vse": "vse_syllabi_raw_parsed.csv",
    "bi": "bi_syllabi_raw_parsed.csv",
    "canterbury": "canterbury_syllabi_raw_parsed.csv",
    "edinburgh": "edinburgh_syllabi_raw_parsed.csv",
    "hkust": "hkust_syllabi_raw_parsed.csv",
    "itam": "itam_syllabi_raw_parser.csv",
    "queens": "queens_syllabi_raw_parsed.csv"
}

# načtení datasetů do dictionary
dfs = {}

for name, filename in DATASETS.items():

    file_path = INTERIM_DIR / filename

    df = pd.read_csv(file_path)

    # kontrola shody schématu
    missing_columns = set(FINAL_COLUMNS) - set(df.columns)
    extra_columns = set(df.columns) - set(FINAL_COLUMNS)

    print(f"\n{name.upper()}")

    if missing_columns:
        print(f"Chybějící sloupce: {missing_columns}")

    if extra_columns:
        print(f"Nadbytečné sloupce: {extra_columns}")

    if not missing_columns and not extra_columns:
        print("Schéma odpovídá.")

    # seřazení sloupců podle jednotného schématu
    df = df[FINAL_COLUMNS]

    dfs[name] = df


VSE
Schéma odpovídá.

BI
Schéma odpovídá.

CANTERBURY
Schéma odpovídá.

EDINBURGH
Schéma odpovídá.

HKUST
Schéma odpovídá.

ITAM
Schéma odpovídá.

QUEENS
Schéma odpovídá.


## Vytvoření hlavních datasetů

In [3]:
# referenční dataset sylabů VŠE
vse_df = dfs["vse"].copy()

# spojení datasetů partnerských univerzit
partner_df = pd.concat(
    [
        dfs["bi"],
        dfs["canterbury"],
        dfs["edinburgh"],
        dfs["hkust"],
        dfs["itam"],
        dfs["queens"]
    ],
    ignore_index=True
)

# kontrola velikosti datasetů
print(f"VŠE dataset: {len(vse_df)} záznamů")
print(f"Partnerský dataset: {len(partner_df)} záznamů")

VŠE dataset: 133 záznamů
Partnerský dataset: 936 záznamů


## Základní přehled hlavních datasetů

In [4]:
# základní přehled hlavních datasetů
overview_df = pd.DataFrame([
    {
        "dataset": "vse",
        "rows": len(vse_df),
        "columns": len(vse_df.columns),
        "missing_course_title": vse_df["course_title"].isna().sum(),
        "missing_course_code": vse_df["course_code"].isna().sum()
    },
    {
        "dataset": "partners",
        "rows": len(partner_df),
        "columns": len(partner_df.columns),
        "missing_course_title": partner_df["course_title"].isna().sum(),
        "missing_course_code": partner_df["course_code"].isna().sum()
    }
])

overview_df

,dataset,rows,columns,missing_course_title,missing_course_code
0,vse,133,17,0,0
1,partners,936,17,0,0


## Základní sjednocení textových polí

In [5]:
# textové sloupce datasetu
TEXT_COLUMNS = [
    "course_title",
    "course_code",
    "university",
    "country",
    "source_url",
    "academic_year",
    "semester",
    "semester_normalized",
    "study_level",
    "credit_system",
    "aims",
    "course_content",
    "learning_outcomes",
    "assessment_methods",
    "literature"
]

# základní technické čištění textových polí
for df in [vse_df, partner_df]:

    for column in TEXT_COLUMNS:

        # převod na string datatype
        df[column] = df[column].astype("string")

        # odstranění nadbytečných whitespace znaků
        df[column] = (
            df[column]
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

## Kontrola sloupce `course_title`

In [6]:
# základní kontrola názvů předmětů
course_title_summary = pd.DataFrame([
    {
        "dataset": "vse",
        "missing_values": vse_df["course_title"].isna().sum()
    },
    {
        "dataset": "partners",
        "missing_values": partner_df["course_title"].isna().sum()
    }
])

course_title_summary

,dataset,missing_values
0,vse,0
1,partners,0


## Kontrola sloupce `course_code`

In [7]:
# základní kontrola kódů předmětů
course_code_summary = pd.DataFrame([
    {
        "dataset": "vse",
        "missing_values": vse_df["course_code"].isna().sum(),
        "unique_values": vse_df["course_code"].nunique(),
        "duplicated_values": vse_df["course_code"].duplicated().sum()
    },
    {
        "dataset": "partners",
        "missing_values": partner_df["course_code"].isna().sum(),
        "unique_values": partner_df["course_code"].nunique(),
        "duplicated_values": partner_df["course_code"].duplicated().sum()
    }
])

course_code_summary

,dataset,missing_values,unique_values,duplicated_values
0,vse,0,133,0
1,partners,0,903,33


In [8]:
# počet duplicitních kódů podle univerzit
duplicate_course_codes_summary = (
    partner_df[partner_df["course_code"].duplicated(keep=False)]
    .groupby("university")["course_code"]
    .nunique()
    .reset_index(name="duplicate_course_codes")
    .sort_values("duplicate_course_codes", ascending=False)
)

duplicate_course_codes_summary

,university,duplicate_course_codes
0,Queen's University,32
1,University of Edinburgh,1


In [9]:
# kontrola úplných duplicitních řádků podle univerzit
full_duplicate_rows = (
    partner_df[partner_df.duplicated(keep=False)]
    .groupby("university")
    .size()
    .reset_index(name="full_duplicate_rows")
    .sort_values("full_duplicate_rows", ascending=False)
)

full_duplicate_rows

,university,full_duplicate_rows
0,University of Edinburgh,2


In [10]:
# kontrola duplicit na Queen's University po ignorování semestru
queens_df = partner_df[
    partner_df["university"] == "Queen's University"
].copy()

compare_columns_without_semester = [
    column for column in FINAL_COLUMNS
    if column not in ["semester", "semester_normalized"]
]

queens_duplicates_without_semester = (
    queens_df
    .drop(columns=["semester", "semester_normalized"])
    .duplicated()
    .sum()
)

queens_problematic_codes = (
    queens_df
    .groupby("course_code")[compare_columns_without_semester]
    .nunique(dropna=False)
)

queens_problematic_codes = queens_problematic_codes[
    (queens_problematic_codes > 1).any(axis=1)
].index.tolist()

print(
    f"Počet duplicitních řádků na Queen's po ignorování semestru: "
    f"{queens_duplicates_without_semester}"
)

print(
    f"Kódy s rozdílem i mimo semestr: {queens_problematic_codes}"
)

Počet duplicitních řádků na Queen's po ignorování semestru: 31
Kódy s rozdílem i mimo semestr: ['COMM 321']


In [11]:
# kontrola rozdílů u výjimky COMM 321
comm_321 = queens_df[
    queens_df["course_code"] == "COMM 321"
].copy()

comm_321_diff = (
    comm_321[compare_columns_without_semester]
    .nunique(dropna=False)
    .reset_index()
)

comm_321_diff.columns = ["column", "unique_values"]

comm_321_diff[
    comm_321_diff["unique_values"] > 1
]

,column,unique_values
0,course_title,2


In [12]:
# hodnoty rozdílného názvu u COMM 321
comm_321[
    ["course_code", "course_title", "semester", "semester_normalized"]
]

,course_code,course_title,semester,semester_normalized
850,COMM 321,Derivative Securities (equivalent to ),Fall,winter
903,COMM 321,Derivative Securities,Winter,summer


### Rozhodnutí k duplicitám ve sloupci `course_code`

Kontrola ukázala, že úplná duplicita řádku se vyskytuje u University of Edinburgh. U Queen's University jsou duplicity způsobeny tím, že stejné předměty jsou uvedeny pro semestry Fall a Winter. Po ignorování sloupců `semester` a `semester_normalized` jsou záznamy shodné. Výjimkou je předmět `COMM 321`, u kterého se mimo semestr liší pouze hodnota ve sloupci `course_title`. Jelikož jde o stejný kód předmětu a název se liší jen doplňující poznámkou v jedné z variant, byl i tento případ vyhodnocen jako duplicitní záznam. Vzhledem k tomu, že cílem datové sady je reprezentovat sylabus předmětu, nikoliv každé semestrální vypsání zvlášť, bude pro Queen's University ponechána pouze varianta Winter, která odpovídá novější verzi v akademickém roce 2025/2026.

In [13]:
# odstranění úplných duplicitních řádků
partner_df = partner_df.drop_duplicates().reset_index(drop=True)

# odstranění duplicit u Queen's University se zachováním varianty Winter
queens_mask = partner_df["university"] == "Queen's University"

queens_clean = (
    partner_df[queens_mask]
    .assign(
        semester_priority=lambda df: df["semester"].eq("Winter").astype(int)
    )
    .sort_values(
        by=["course_code", "semester_priority"],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset=["course_code"],
        keep="first"
    )
    .drop(columns=["semester_priority"])
)

# složení partnerského datasetu po deduplikaci
partner_df = pd.concat(
    [
        partner_df[~queens_mask],
        queens_clean
    ],
    ignore_index=True
)

# kontrola duplicit po čištění
partner_df[
    partner_df.duplicated(
        subset=["university", "course_code"],
        keep=False
    )
]

,course_title,course_code,university,country,source_url,academic_year,semester,semester_normalized,study_level,credits,credit_system,ects_credits,aims,course_content,learning_outcomes,assessment_methods,literature


## Kontrola sloupce `university`

In [14]:
# kontrola názvů univerzit
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["university"].isna().sum())

print("Počet unikátních hodnot:",
      vse_df["university"].nunique())

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["university"].isna().sum())

print("Počet unikátních hodnot:",
      partner_df["university"].nunique())

print("\nUniverzity v partnerském datasetu:")

sorted(partner_df["university"].unique())

VŠE
Chybějící hodnoty: 0
Počet unikátních hodnot: 1

Partneři
Chybějící hodnoty: 0
Počet unikátních hodnot: 6

Univerzity v partnerském datasetu:


['BI Norwegian Business School',
 'Instituto Tecnológico Autónomo de México',
 "Queen's University",
 'The Hong Kong University of Science and Technology',
 'University of Canterbury',
 'University of Edinburgh']

## Kontrola sloupce `country`

In [15]:
# kontrola názvů zemí
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["country"].isna().sum())

print("Počet unikátních hodnot:",
      vse_df["country"].nunique())

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["country"].isna().sum())

print("Počet unikátních hodnot:",
      partner_df["country"].nunique())

print("\nZemě v partnerském datasetu:")

sorted(partner_df["country"].unique())

VŠE
Chybějící hodnoty: 0
Počet unikátních hodnot: 1

Partneři
Chybějící hodnoty: 0
Počet unikátních hodnot: 6

Země v partnerském datasetu:


['Canada', 'Hong Kong', 'Mexico', 'New Zealand', 'Norway', 'United Kingdom']

## Kontrola sloupce `source_url`

In [16]:
# kontrola URL adres zdrojů
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["source_url"].isna().sum())

print("Počet unikátních hodnot:",
      vse_df["source_url"].nunique())

print("Duplicitní hodnoty:",
      vse_df["source_url"].duplicated().sum())

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["source_url"].isna().sum())

print("Počet unikátních hodnot:",
      partner_df["source_url"].nunique())

print("Duplicitní hodnoty:",
      partner_df["source_url"].duplicated().sum())

print("\nDuplicitní URL podle univerzit:")

(
    partner_df[
        partner_df["source_url"].duplicated(keep=False)
    ]
    .groupby("university")["source_url"]
    .nunique()
    .reset_index(name="duplicate_source_urls")
    .sort_values("duplicate_source_urls", ascending=False)
)

VŠE
Chybějící hodnoty: 0
Počet unikátních hodnot: 133
Duplicitní hodnoty: 0

Partneři
Chybějící hodnoty: 0
Počet unikátních hodnot: 501
Duplicitní hodnoty: 402

Duplicitní URL podle univerzit:


,university,duplicate_source_urls
1,The Hong Kong University of Science and Technology,12
0,Instituto Tecnológico Autónomo de México,1


In [17]:
# výpis duplicitních URL u HKUST
sorted(
    partner_df[
        (
            partner_df["university"] ==
            "The Hong Kong University of Science and Technology"
        )
        &
        (
            partner_df["source_url"].duplicated(keep=False)
        )
    ]["source_url"]
    .drop_duplicates()
    .tolist()
)

['https://prog-crs.hkust.edu.hk/ugcourse/2025-26/ACCT',
 'https://prog-crs.hkust.edu.hk/ugcourse/2025-26/COMP',
 'https://prog-crs.hkust.edu.hk/ugcourse/2025-26/DASC',
 'https://prog-crs.hkust.edu.hk/ugcourse/2025-26/ECON',
 'https://prog-crs.hkust.edu.hk/ugcourse/2025-26/ENTR',
 'https://prog-crs.hkust.edu.hk/ugcourse/2025-26/FINA',
 'https://prog-crs.hkust.edu.hk/ugcourse/2025-26/GBUS',
 'https://prog-crs.hkust.edu.hk/ugcourse/2025-26/ISOM',
 'https://prog-crs.hkust.edu.hk/ugcourse/2025-26/MARK',
 'https://prog-crs.hkust.edu.hk/ugcourse/2025-26/MATH',
 'https://prog-crs.hkust.edu.hk/ugcourse/2025-26/MGMT',
 'https://prog-crs.hkust.edu.hk/ugcourse/2025-26/RMBI']

U sloupce `source_url` byly identifikovány duplicitní hodnoty zejména u Instituto Tecnológico Autónomo de México, kde všechny předměty odkazují na společný zdrojový dokument obsahující více sylabů. Menší počet duplicit se vyskytoval také u The Hong Kong University of Science and Technology. Kontrola ukázala, že tyto URL odpovídají stránkám jednotlivých předmětových oblastí definovaných prefixem kódu předmětu (např. ACCT, COMP, ISOM nebo MATH). Jednotlivé předměty byly na těchto stránkách dostupné jako rozbalitelné sekce v rámci jedné společné URL adresy, nikoliv jako samostatné detailní stránky. Duplicitní URL adresy proto byly vyhodnoceny jako očekávaná vlastnost zdrojových dat a nebyly dále upravovány.

## Kontrola sloupce `academic_year`

In [18]:
# kontrola akademických roků
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["academic_year"].isna().sum())

print("Počet unikátních hodnot:",
      vse_df["academic_year"].nunique())

print("\nPočty řádků podle academic_year:")

print(
    vse_df["academic_year"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["academic_year"].isna().sum())

print("Počet unikátních hodnot:",
      partner_df["academic_year"].nunique())

print("\nPočty řádků podle academic_year:")

print(
    partner_df["academic_year"]
    .value_counts(dropna=False)
    .sort_index()
)

VŠE
Chybějící hodnoty: 0
Počet unikátních hodnot: 2

Počty řádků podle academic_year:
academic_year
2025/2026    55
2026/2027    78
Name: count, dtype: Int64

Partneři
Chybějící hodnoty: 0
Počet unikátních hodnot: 13

Počty řádků podle academic_year:
academic_year
2015-2016      1
2016-2017      3
2018-2019      1
2019           1
2019-2020      2
2023-2024      3
2024          39
2024-2025     22
2025          49
2025-2026     71
2025-26      382
2025/26       98
2026         231
Name: count, dtype: Int64


### Sjednocení formátu akademického roku

In [19]:
# sjednocení formátu academic_year
def normalize_academic_year(row):

    year = str(row["academic_year"]).strip()

    semester = (
        str(row["semester_normalized"])
        .strip()
        .lower()
    )

    # formát 2025-2026 nebo 2025/2026
    match_full = re.match(
        r"^(\d{4})[-/](\d{4})$",
        year
    )

    if match_full:

        return (
            f"{match_full.group(1)}"
            f"/{match_full.group(2)}"
        )

    # formát 2025-26 nebo 2025/26
    match_short = re.match(
        r"^(\d{4})[-/](\d{2})$",
        year
    )

    if match_short:

        start_year = int(match_short.group(1))

        end_year = int(
            str(start_year)[:2] +
            match_short.group(2)
        )

        return f"{start_year}/{end_year}"

    # formát s jedním rokem
    match_single = re.match(
        r"^(\d{4})$",
        year
    )

    if match_single:

        year_value = int(match_single.group(1))

        # winter = začátek akademického roku
        if semester == "winter":

            return (
                f"{year_value}"
                f"/{year_value + 1}"
            )

        # summer = konec akademického roku
        if semester == "summer":

            return (
                f"{year_value - 1}"
                f"/{year_value}"
            )

    # ostatní hodnoty ponecháme beze změny
    return year

# aplikace normalizace
vse_df["academic_year"] = (
    vse_df.apply(
        normalize_academic_year,
        axis=1
    )
)

partner_df["academic_year"] = (
    partner_df.apply(
        normalize_academic_year,
        axis=1
    )
)

# kontrola výsledných hodnot
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["academic_year"].isna().sum())

print("Počet unikátních hodnot:",
      vse_df["academic_year"].nunique())

print("\nPočty řádků podle academic_year:")

print(
    vse_df["academic_year"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["academic_year"].isna().sum())

print("Počet unikátních hodnot:",
      partner_df["academic_year"].nunique())

print("\nPočty řádků podle academic_year:")

print(
    partner_df["academic_year"]
    .value_counts(dropna=False)
    .sort_index()
)

VŠE
Chybějící hodnoty: 0
Počet unikátních hodnot: 2

Počty řádků podle academic_year:
academic_year
2025/2026    55
2026/2027    78
Name: count, dtype: int64

Partneři
Chybějící hodnoty: 0
Počet unikátních hodnot: 9

Počty řádků podle academic_year:
academic_year
2015/2016      1
2016/2017      3
2018/2019      1
2019           1
2019/2020      2
2023/2024      3
2024/2025     76
2025/2026    690
2026/2027    126
Name: count, dtype: int64


Hodnoty ve sloupci `academic_year` byly sjednoceny do formátu `YYYY/YYYY`. U hodnot obsahujících pouze jeden rok byl akademický rok dopočten pouze v případech, kdy byl dostupný normalizovaný semestr. Hodnoty bez dostatečného kontextu nebyly automaticky upravovány.

## Kontrola sloupce `semester`

In [20]:
# kontrola hodnot semester
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["semester"].isna().sum())

print("Počet unikátních hodnot:",
      vse_df["semester"].nunique())

print("\nPočty řádků podle semester:")

print(
    vse_df["semester"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["semester"].isna().sum())

print("Počet unikátních hodnot:",
      partner_df["semester"].nunique())

print("\nPočty řádků podle semester:")

print(
    partner_df["semester"]
    .value_counts(dropna=False)
    .sort_index()
)

VŠE
Chybějící hodnoty: 0
Počet unikátních hodnot: 2

Počty řádků podle semester:
semester
WS 2025/2026    55
WS 2026/2027    78
Name: count, dtype: Int64

Partneři
Chybějící hodnoty: 384
Počet unikátních hodnot: 11

Počty řádků podle semester:
semester
Autumn           73
Fall             58
Full Year         2
Semester 1       44
Semester 2       52
Semester One    104
Semester Two     88
Spring           16
Winter           51
fall             26
spring            5
<NA>            384
Name: count, dtype: Int64


In [21]:
# sjednocení velikosti písmen u semester
partner_df["semester"] = (
    partner_df["semester"]
    .replace({
        "fall": "Fall",
        "spring": "Spring"
    })
)

# kontrola výsledných hodnot
partner_df["semester"].value_counts(
    dropna=False
).sort_index()

semester
Autumn           73
Fall             84
Full Year         2
Semester 1       44
Semester 2       52
Semester One    104
Semester Two     88
Spring           21
Winter           51
<NA>            384
Name: count, dtype: Int64

U sloupce `semester` byly sjednoceny pouze rozdíly ve velikosti písmen u hodnot `Fall` a `Spring`. Ostatní varianty byly ponechány v podobě publikované zdrojovou institucí.

## Kontrola sloupce `semester_normalized`

In [22]:
# kontrola hodnot semester_normalized
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["semester_normalized"].isna().sum())

print("Počet unikátních hodnot:",
      vse_df["semester_normalized"].nunique())

print("\nPočty řádků podle semester_normalized:")

print(
    vse_df["semester_normalized"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["semester_normalized"].isna().sum())

print("Počet unikátních hodnot:",
      partner_df["semester_normalized"].nunique())

print("\nPočty řádků podle semester_normalized:")

print(
    partner_df["semester_normalized"]
    .value_counts(dropna=False)
    .sort_index()
)

VŠE
Chybějící hodnoty: 0
Počet unikátních hodnot: 1

Počty řádků podle semester_normalized:
semester_normalized
winter    133
Name: count, dtype: Int64

Partneři
Chybějící hodnoty: 384
Počet unikátních hodnot: 3

Počty řádků podle semester_normalized:
semester_normalized
full_year      2
summer       228
winter       289
<NA>         384
Name: count, dtype: Int64


Sloupec `semester_normalized` obsahuje sjednocenou reprezentaci semestru napříč institucemi. Hodnoty byly mapovány na kategorie `winter`, `summer` a `full_year`. Chybějící hodnoty odpovídají případům, kdy zdrojová data informaci o semestru neobsahovala.

## Kontrola sloupce `study_level`

In [23]:
# kontrola hodnot study_level
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["study_level"].isna().sum())

print("Počet unikátních hodnot:",
      vse_df["study_level"].nunique())

print("\nPočty řádků podle study_level:")

print(
    vse_df["study_level"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["study_level"].isna().sum())

print("Počet unikátních hodnot:",
      partner_df["study_level"].nunique())

print("\nPočty řádků podle study_level:")

print(
    partner_df["study_level"]
    .value_counts(dropna=False)
    .sort_index()
)

VŠE
Chybějící hodnoty: 0
Počet unikátních hodnot: 2

Počty řádků podle study_level:
study_level
bachelor    124
master        9
Name: count, dtype: Int64

Partneři
Chybějící hodnoty: 0
Počet unikátních hodnot: 4

Počty řádků podle study_level:
study_level
Bachelor          56
Master            71
Undergraduate     98
undergraduate    678
Name: count, dtype: Int64


In [24]:
# sjednocení velikosti písmen u study_level
STUDY_LEVEL_MAPPING = {
    "Bachelor": "bachelor",
    "Master": "master",
    "Undergraduate": "undergraduate"
}

vse_df["study_level"] = (
    vse_df["study_level"]
    .replace(STUDY_LEVEL_MAPPING)
)

partner_df["study_level"] = (
    partner_df["study_level"]
    .replace(STUDY_LEVEL_MAPPING)
)

# kontrola výsledných hodnot
print("VŠE")

print(
    vse_df["study_level"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPartneři")

print(
    partner_df["study_level"]
    .value_counts(dropna=False)
    .sort_index()
)

VŠE
study_level
bachelor    124
master        9
Name: count, dtype: Int64

Partneři
study_level
bachelor          56
master            71
undergraduate    776
Name: count, dtype: Int64


U sloupce `study_level` byly sjednoceny pouze rozdíly ve velikosti písmen. Hodnoty reprezentující terminologii jednotlivých vzdělávacích systémů byly ponechány beze změny.

## Kontrola sloupce `credits`

In [25]:
# kontrola hodnot credits
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["credits"].isna().sum())

print("Počet unikátních hodnot:",
      vse_df["credits"].nunique())

print("\nPočty řádků podle credits:")

print(
    vse_df["credits"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["credits"].isna().sum())

print("Počet unikátních hodnot:",
      partner_df["credits"].nunique())

print("\nPočty řádků podle credits:")

print(
    partner_df["credits"]
    .value_counts(dropna=False)
    .sort_index()
)

VŠE
Chybějící hodnoty: 0
Počet unikátních hodnot: 6

Počty řádků podle credits:
credits
2      2
3     12
4     17
5     31
6     70
30     1
Name: count, dtype: int64

Partneři
Chybějící hodnoty: 0
Počet unikátních hodnot: 13

Počty řádků podle credits:
credits
0.0       1
1.0      38
2.0      12
3.0     318
4.0      80
5.0       1
6.0      68
7.5      91
8.0       2
15.0    193
20.0     96
30.0      1
40.0      2
Name: count, dtype: int64


In [26]:
# kontrola nestandardních hodnot credits
partner_df[
    partner_df["credits"].isin([0.0, 40.0])
][
    [
        "university",
        "course_code",
        "course_title",
        "credits",
        "credit_system",
        "ects_credits",
        "source_url"
    ]
]

,university,course_code,course_title,credits,credit_system,ects_credits,source_url
49,BI Norwegian Business School,FORK 2200,Preparatory Course in Programming,0.0,ECTS,0.0,https://programmeinfo.bi.no/nb/course/FORK-2200/2026-autumn
383,University of Edinburgh,ECNM08013,Economics 1,40.0,SCQF credits,20.0,https://www.drps.ed.ac.uk/25-26/dpt/cxecnm08013.htm
385,University of Edinburgh,ECNM08006,Economics 2,40.0,SCQF credits,20.0,https://www.drps.ed.ac.uk/25-26/dpt/cxecnm08006.htm


In [27]:
# odstranění nekreditovaného přípravného kurzu
partner_df = partner_df[
    partner_df["credits"] != 0
].reset_index(drop=True)

# sjednocení datového typu credits
vse_df["credits"] = (
    vse_df["credits"]
    .astype(float)
)

partner_df["credits"] = (
    partner_df["credits"]
    .astype(float)
)

# kontrola výsledných hodnot
print("VŠE")

print(
    vse_df["credits"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPartneři")

print(
    partner_df["credits"]
    .value_counts(dropna=False)
    .sort_index()
)

VŠE
credits
2.0      2
3.0     12
4.0     17
5.0     31
6.0     70
30.0     1
Name: count, dtype: int64

Partneři
credits
1.0      38
2.0      12
3.0     318
4.0      80
5.0       1
6.0      68
7.5      91
8.0       2
15.0    193
20.0     96
30.0      1
40.0      2
Name: count, dtype: int64


U partnerských univerzit byl identifikován jeden předmět s hodnotou `0` ve sloupci `credits`. Kontrola zdrojového sylabu ukázala, že se jednalo o přípravný nekreditovaný kurz, nikoliv standardní akademický předmět. Záznam byl proto z datasetu odstraněn.

## Kontrola sloupce `credit_system`

In [28]:
# kontrola hodnot credit_system
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["credit_system"].isna().sum())

print("Počet unikátních hodnot:",
      vse_df["credit_system"].nunique())

print("\nPočty řádků podle credit_system:")

print(
    vse_df["credit_system"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["credit_system"].isna().sum())

print("Počet unikátních hodnot:",
      partner_df["credit_system"].nunique())

print("\nPočty řádků podle credit_system:")

print(
    partner_df["credit_system"]
    .value_counts(dropna=False)
    .sort_index()
)

VŠE
Chybějící hodnoty: 0
Počet unikátních hodnot: 1

Počty řádků podle credit_system:
credit_system
ECTS    133
Name: count, dtype: Int64

Partneři
Chybějící hodnoty: 0
Počet unikátních hodnot: 6

Počty řádků podle credit_system:
credit_system
ECTS             126
HKUST credits    382
ITAM credits      33
Queen's units     71
SCQF credits      98
UC points        192
Name: count, dtype: Int64


Sloupec `credit_system` byl ponechán beze změny. Hodnoty odpovídají kreditovým systémům používaným jednotlivými institucemi.

## Kontrola sloupce `ects_credits`

In [29]:
# kontrola hodnot ects_credits
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["ects_credits"].isna().sum())

print("Počet unikátních hodnot:",
      vse_df["ects_credits"].nunique())

print("\nPočty řádků podle ects_credits:")

print(
    vse_df["ects_credits"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["ects_credits"].isna().sum())

print("Počet unikátních hodnot:",
      partner_df["ects_credits"].nunique())

print("\nPočty řádků podle ects_credits:")

print(
    partner_df["ects_credits"]
    .value_counts(dropna=False)
    .sort_index()
)

VŠE
Chybějící hodnoty: 0
Počet unikátních hodnot: 6

Počty řádků podle ects_credits:
ects_credits
2      2
3     12
4     17
5     31
6     70
30     1
Name: count, dtype: int64

Partneři
Chybějící hodnoty: 0
Počet unikátních hodnot: 11

Počty řádků podle ects_credits:
ects_credits
2.0      38
4.0      12
5.0      31
6.0     351
7.0       2
7.5     282
8.0      80
10.0     97
12.0      4
15.0      3
20.0      2
Name: count, dtype: int64


In [30]:
# sjednocení datového typu ects_credits
vse_df["ects_credits"] = (
    vse_df["ects_credits"]
    .astype(float)
)

partner_df["ects_credits"] = (
    partner_df["ects_credits"]
    .astype(float)
)

# kontrola datových typů
print(vse_df["ects_credits"].dtype)

print(partner_df["ects_credits"].dtype)

float64
float64


In [31]:
# kontrola správnosti přepočtu na ECTS kredity
credit_check_df = partner_df.copy()

credit_check_df["credits_num"] = pd.to_numeric(
    credit_check_df["credits"],
    errors="coerce"
)

credit_check_df["ects_num"] = pd.to_numeric(
    credit_check_df["ects_credits"],
    errors="coerce"
)

def check_ects_conversion(row):

    university = row["university"]
    credits = row["credits_num"]
    ects = row["ects_num"]

    if pd.isna(credits) or pd.isna(ects):
        return False

    if university == "BI Norwegian Business School":
        expected = credits

    elif university == "The Hong Kong University of Science and Technology":
        expected = credits * 2

    elif university == "Instituto Tecnológico Autónomo de México":
        expected = round(credits * 5 / 6)

    elif university == "University of Canterbury":
        expected = credits / 2

    elif university == "Queen's University":
        expected = credits * 2

    elif university == "University of Edinburgh":
        expected = credits / 2

    else:
        return False

    return abs(expected - ects) < 0.01

credit_check_df["ects_conversion_correct"] = (
    credit_check_df
    .apply(check_ects_conversion, axis=1)
)

print("Počet správných přepočtů:")
print(
    credit_check_df["ects_conversion_correct"]
    .sum()
)

print("\nPočet nesprávných přepočtů:")
print(
    (~credit_check_df["ects_conversion_correct"])
    .sum()
)

print("\nNesprávné přepočty:")

display(
    credit_check_df[
        credit_check_df["ects_conversion_correct"] == False
    ][
        [
            "course_code",
            "course_title",
            "university",
            "credits",
            "ects_credits",
            "source_url"
        ]
    ]
)

Počet správných přepočtů:
902

Počet nesprávných přepočtů:
0

Nesprávné přepočty:


,course_code,course_title,university,credits,ects_credits,source_url


Kontrola ukázala, že hodnoty ve sloupci `ects_credits` odpovídají převodním pravidlům definovaným pro jednotlivé kreditové systémy partnerských univerzit. Nebyly identifikovány žádné nekonzistentní přepočty.

## Kontrola sloupce `aims`

In [32]:
# kontrola hodnot aims
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["aims"].isna().sum())

print("Prázdné řetězce:",
      (vse_df["aims"].str.strip() == "").sum())

print("\nDélka textu:")

print(
    vse_df["aims"]
    .str.len()
    .describe()
)

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["aims"].isna().sum())

print("Prázdné řetězce:",
      (partner_df["aims"].str.strip() == "").sum())

print("\nDélka textu:")

print(
    partner_df["aims"]
    .str.len()
    .describe()
)

VŠE
Chybějící hodnoty: 0
Prázdné řetězce: 0

Délka textu:
count         133.0
mean     347.180451
std      281.991184
min            22.0
25%           132.0
50%           304.0
75%           450.0
max          1526.0
Name: aims, dtype: Float64

Partneři
Chybějící hodnoty: 2
Prázdné řetězce: 0

Délka textu:
count         900.0
mean     549.474444
std      426.468519
min            26.0
25%          257.75
50%           462.0
75%          699.25
max          4931.0
Name: aims, dtype: Float64


In [33]:
# nejčastější začátky textů ve sloupci aims
print("VŠE")

display(
    vse_df["aims"]
    .str[:80]
    .value_counts()
    .head(15)
)

print("\nPartneři")

display(
    partner_df["aims"]
    .str[:80]
    .value_counts()
    .head(15)
)

VŠE


aims
The aims of the course has not been saved in this language version.                 27
-- item not defined --                                                               3
Accounting as an information tool for the decision-making of an entity’s owners,     3
The course focuses on the process of finding entrepreneurial opportunities, veri     2
The purpose is to teach students to think economically, to familiarize students      2
Students will learn the purpose of a business plan, its users, methods, techniqu     1
The aim of the course is that students get to know to the issue of marketing and     1
A good knowledge of the office pack is useful for students both for elaboration      1
With the growing integration of Czech enterprises into international cooperation     1
Students will get to know the possibilities of database systems, which are a par     1
Team collaboration software tools are playing increasingly important role in eve     1
The course is focused on the basics of


Partneři


aims
This special topic will allow flexibility to offer new or one-off courses of str    4
This is a project-based course that provides students an opportunity to integrat    4
Individual Course of Study                                                          3
Analysis of sequentially collected data including data modelling and forecasting    3
This intermediate level course is intended to develop a rigorous understanding o    2
This course is a supervised course which trains student to do proper economic re    2
Financial technology (Fintech) refers to the digitalization and technological in    2
An introduction to the ideas, techniques and applications of statistics and prob    2
The course covers basic applications and techniques of single-variable calculus,    2
The overall goal of the course it to provide the participants with a sound under    2
The MATH 2043 and 3043 is a rigorous sequence in analysis on the line and higher    2
Basic elements of starting a new business in info

In [34]:
# sjednocení placeholder hodnot ve sloupci aims
AIMS_PLACEHOLDERS = [
    "The aims of the course has not been saved in this language version.",
    "-- item not defined --"
]

vse_df["aims"] = (
    vse_df["aims"]
    .replace(AIMS_PLACEHOLDERS, pd.NA)
)

# kontrola výsledku
print("VŠE")

print(
    vse_df["aims"]
    .isna()
    .sum()
)

print("\nPartneři")

print(
    partner_df["aims"]
    .isna()
    .sum()
)

VŠE
30

Partneři
2


In [35]:
# kontrola hodnoty "Individual Course of Study"
partner_df[
    partner_df["aims"]
    .str.contains(
        "Individual Course of Study",
        na=False
    )
][
    [
        "university",
        "course_code",
        "course_title",
        "aims",
        "source_url"
    ]
]

,university,course_code,course_title,aims,source_url
248,University of Canterbury,MATH393,Individual Course of Study,Individual Course of Study,https://courseinfo.canterbury.ac.nz/GetCourseDetails.aspx?course=MATH393&occurrence=26S1(C)&year=2026
249,University of Canterbury,MATH394,Individual Course of Study,Individual Course of Study,https://courseinfo.canterbury.ac.nz/GetCourseDetails.aspx?course=MATH394&occurrence=26S2(C)&year=2026
317,University of Canterbury,STAT394,Individual Course of Study,Individual Course of Study,https://courseinfo.canterbury.ac.nz/GetCourseDetails.aspx?course=STAT394&occurrence=26S2(C)&year=2026


In [36]:
# kontrola záznamů s hodnotou "Individual Course of Study"
individual_courses = partner_df[
    partner_df["aims"].eq("Individual Course of Study")
].copy()

individual_courses[
    [
        "university",
        "course_code",
        "course_title",
        "credits",
        "ects_credits",
        "aims",
        "course_content",
        "learning_outcomes",
        "assessment_methods",
        "literature",
        "source_url"
    ]
]

,university,course_code,course_title,credits,ects_credits,aims,course_content,learning_outcomes,assessment_methods,literature,source_url
248,University of Canterbury,MATH393,Individual Course of Study,15.0,7.5,Individual Course of Study,<NA>,<NA>,<NA>,<NA>,https://courseinfo.canterbury.ac.nz/GetCourseDetails.aspx?course=MATH393&occurrence=26S1(C)&year=2026
249,University of Canterbury,MATH394,Individual Course of Study,15.0,7.5,Individual Course of Study,<NA>,<NA>,<NA>,<NA>,https://courseinfo.canterbury.ac.nz/GetCourseDetails.aspx?course=MATH394&occurrence=26S2(C)&year=2026
317,University of Canterbury,STAT394,Individual Course of Study,15.0,7.5,Individual Course of Study,<NA>,<NA>,<NA>,<NA>,https://courseinfo.canterbury.ac.nz/GetCourseDetails.aspx?course=STAT394&occurrence=26S2(C)&year=2026


In [37]:
# odstranění neanalyzovatelných záznamů typu Individual Course of Study
partner_df = partner_df[
    ~partner_df["course_code"].isin([
        "MATH393",
        "MATH394",
        "STAT394"
    ])
].reset_index(drop=True)

# kontrola odstranění
partner_df[
    partner_df["course_code"].isin([
        "MATH393",
        "MATH394",
        "STAT394"
    ])
]

,course_title,course_code,university,country,source_url,academic_year,semester,semester_normalized,study_level,credits,credit_system,ects_credits,aims,course_content,learning_outcomes,assessment_methods,literature


In [38]:
# kontrola aims po čištění
print("VŠE")

print(
    vse_df["aims"]
    .isna()
    .sum()
)

print("\nPartneři")

print(
    partner_df["aims"]
    .isna()
    .sum()
)

print("\nNejčastější začátky aims u partnerů:")

display(
    partner_df["aims"]
    .dropna()
    .str[:80]
    .value_counts()
    .head(10)
)

VŠE
30

Partneři
2

Nejčastější začátky aims u partnerů:


aims
This special topic will allow flexibility to offer new or one-off courses of str    4
This is a project-based course that provides students an opportunity to integrat    4
Analysis of sequentially collected data including data modelling and forecasting    3
This course analyses the nature and problem of risk and uncertainty and consumer    2
This calculus-based course is intended to develop a rigorous understanding of co    2
The course covers basic applications and techniques of single-variable calculus,    2
An introduction to the ideas, techniques and applications of statistics and prob    2
An internship or consultancy project is an opportunity to experience a professio    2
The overall goal of the course it to provide the participants with a sound under    2
This course is a supervised course which trains student to do proper economic re    2
Name: count, dtype: Int64

Ve sloupci `aims` byly u datasetu VŠE identifikovány systémové placeholder hodnoty `The aims of the course has not been saved in this language version.` a `-- item not defined --`, které neobsahovaly skutečný obsah sylabu. Tyto hodnoty byly převedeny na chybějící hodnoty. Současně byly z partnerského datasetu odstraněny tři záznamy typu `Individual Course of Study`, které po odstranění placeholder hodnot neobsahovaly žádné relevantní obsahové informace pro porovnání sylabů.

## Kontrola sloupce `course_content`

In [39]:
# kontrola hodnot course_content
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["course_content"].isna().sum())

print("Prázdné řetězce:",
      (vse_df["course_content"].str.strip() == "").sum())

print("\nDélka textu:")

print(
    vse_df["course_content"]
    .str.len()
    .describe()
)

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["course_content"].isna().sum())

print("Prázdné řetězce:",
      (partner_df["course_content"].str.strip() == "").sum())

print("\nDélka textu:")

print(
    partner_df["course_content"]
    .str.len()
    .describe()
)

VŠE
Chybějící hodnoty: 0
Prázdné řetězce: 0

Délka textu:
count         133.0
mean     974.819549
std      912.169474
min            22.0
25%           254.0
50%           722.0
75%          1433.0
max          4410.0
Name: course_content, dtype: Float64

Partneři
Chybějící hodnoty: 643
Prázdné řetězce: 0

Délka textu:
count          256.0
mean     1050.789062
std       994.219818
min              1.0
25%           370.75
50%            682.0
75%          1325.25
max           6949.0
Name: course_content, dtype: Float64


In [40]:
# podíl chybějících hodnot course_content podle univerzit
missing_course_content_ratio = (
    partner_df
    .groupby("university")["course_content"]
    .apply(lambda x: x.isna().mean())
    .reset_index(name="missing_ratio")
)

missing_course_content_ratio["missing_ratio"] = (
    missing_course_content_ratio["missing_ratio"] * 100
).round(1)

missing_course_content_ratio.sort_values(
    "missing_ratio",
    ascending=False
)

,university,missing_ratio
2,Queen's University,100.0
3,The Hong Kong University of Science and Technology,100.0
4,University of Canterbury,100.0
0,BI Norwegian Business School,0.8
1,Instituto Tecnológico Autónomo de México,0.0
5,University of Edinburgh,0.0


In [41]:
# nejčastější začátky textů ve sloupci course_content
print("VŠE")

display(
    vse_df["course_content"]
    .dropna()
    .str[:80]
    .value_counts()
    .head(15)
)

print("\nPartneři")

display(
    partner_df["course_content"]
    .dropna()
    .str[:80]
    .value_counts()
    .head(15)
)

VŠE


course_content
The content of the course has not been saved in this language version.              27
-- item not defined --                                                               4
1.Concept of Law, Sources of International, EU and National Law, Legal Systems (     2
1) Introduction to the financial economics, money and financial markets (basic t     2
1. Managerial accounting. Costing of costs and price. Budgeting. 2. Short-term f     1
1.Marketing and commercial communication (allowance 2/0) a. Marketing communicat     1
1.Data analysis (allowance 0/12) a. Office programmes for data analysis. b. Intr     1
1.Project management (allowance 0/14) a. Fundamentals of project management theo     1
1.Basic database notions. (allowance 0/0) 2.Introduction of the Microsoft Office     1
1.The Basics of Office 365 (allowance 0/4) a. Introduction to the subject. b. Of     1
1.Marketing and strategic marketing planning (allowance 4/0) a. Marketing, marke     1
1.Globalization of the world


Partneři


course_content
This calculus-based course is intended to develop a rigorous understanding of co    2
This course is intended to develop a rigorous understanding of core economic mod    2
-                                                                                   2
Examination of strategic financial and non-financial objectives across organizat    1
The course will be divided in a theoretical and a more empirical, or historical,    1
The course consists of three subtopics: Subtopic 1: Financial Accounting Main fi    1
Economics. In order to clarify what lies in the overall goal of acquiring broad     1
Python basics Programming environment. Variables and data types. Input and outpu    1
The course covers the following topics: Loading, cleaning, and exploration of da    1
Introduction to Spreadsheet Modeling. Introduction to Optimization and Linear Pr    1
In-house data External relevant data Extraction of data Decision analysis Manage    1
Introduction, installation of Python, J

In [42]:
# kontrola hodnoty "-"
partner_df[
    partner_df["course_content"] == "-"
][
    [
        "university",
        "course_code",
        "course_title",
        "course_content",
        "learning_outcomes",
        "assessment_methods",
        "literature",
        "source_url"
    ]
]

,university,course_code,course_title,course_content,learning_outcomes,assessment_methods,literature,source_url
94,BI Norwegian Business School,TEM 0420,Strategic Risk Analysis,-,"After finishing this course, the student should have knowledge about: Differing types of risk that organizations fac...",Ordinary examination,<NA>,https://portal.bi.no/kurs/#/course/en/TEM/0420/202620
97,BI Norwegian Business School,TEM 0240,Social Tensions and Connections in a Polarized World,-,"After completion of this course, students should have knowledge of: How identity drives information selection and in...",Ordinary examination,<NA>,https://portal.bi.no/kurs/#/course/en/TEM/0240/202620


In [43]:
# placeholder hodnoty ve sloupci course_content
COURSE_CONTENT_PLACEHOLDERS = [
    "The content of the course has not been saved in this language version.",
    "-- item not defined --",
    "-"
]

vse_df["course_content"] = (
    vse_df["course_content"]
    .replace(COURSE_CONTENT_PLACEHOLDERS, pd.NA)
)

partner_df["course_content"] = (
    partner_df["course_content"]
    .replace(COURSE_CONTENT_PLACEHOLDERS, pd.NA)
)

# kontrola výsledku
print("VŠE")

print(
    vse_df["course_content"]
    .isna()
    .sum()
)

print("\nPartneři")

print(
    partner_df["course_content"]
    .isna()
    .sum()
)

VŠE
31

Partneři
645


Ve sloupci `course_content` byly identifikovány systémové placeholder hodnoty `The content of the course has not been saved in this language version.`, `-- item not defined --` a `-`, které neobsahovaly skutečný obsah sylabu. Tyto hodnoty byly převedeny na chybějící hodnoty.

## Kontrola sloupce `learning_outcomes`

In [44]:
# kontrola hodnot learning_outcomes
print("VŠE")
print("Chybějící hodnoty:",
      vse_df["learning_outcomes"].isna().sum())

print("Prázdné řetězce:",
      (vse_df["learning_outcomes"].str.strip() == "").sum())

print("\nDélka textu:")

print(
    vse_df["learning_outcomes"]
    .str.len()
    .describe()
)

print("\nPartneři")
print("Chybějící hodnoty:",
      partner_df["learning_outcomes"].isna().sum())

print("Prázdné řetězce:",
      (partner_df["learning_outcomes"].str.strip() == "").sum())

print("\nDélka textu:")

print(
    partner_df["learning_outcomes"]
    .str.len()
    .describe()
)

VŠE
Chybějící hodnoty: 0
Prázdné řetězce: 0

Délka textu:
count         133.0
mean      498.81203
std      419.585314
min            22.0
25%           195.0
50%           459.0
75%           651.0
max          3125.0
Name: learning_outcomes, dtype: Float64

Partneři
Chybějící hodnoty: 165
Prázdné řetězce: 0

Délka textu:
count         734.0
mean     722.730245
std      485.374072
min            85.0
25%          412.25
50%           591.0
75%           893.0
max          3951.0
Name: learning_outcomes, dtype: Float64


In [45]:
# podíl chybějících hodnot learning_outcomes podle univerzit
missing_learning_outcomes_ratio = (
    partner_df
    .groupby("university")["learning_outcomes"]
    .apply(lambda x: x.isna().mean())
    .reset_index(name="missing_ratio")
)

missing_learning_outcomes_ratio["missing_ratio"] = (
    missing_learning_outcomes_ratio["missing_ratio"] * 100
).round(1)

missing_learning_outcomes_ratio.sort_values(
    "missing_ratio",
    ascending=False
)

,university,missing_ratio
2,Queen's University,100.0
4,University of Canterbury,43.4
1,Instituto Tecnológico Autónomo de México,33.3
3,The Hong Kong University of Science and Technology,0.3
0,BI Norwegian Business School,0.0
5,University of Edinburgh,0.0


In [46]:
# nejčastější začátky textů ve sloupci learning_outcomes
print("VŠE")

display(
    vse_df["learning_outcomes"]
    .dropna()
    .str[:80]
    .value_counts()
    .head(15)
)

print("\nPartneři")

display(
    partner_df["learning_outcomes"]
    .dropna()
    .str[:80]
    .value_counts()
    .head(15)
)

VŠE


learning_outcomes
The learning outcomes of the course has not been saved in this language version.    27
Upon successful completion of this course, students will be able to: - understan    12
-- item not defined --                                                               3
Upon successful completion of this course, students will be able to: - describe      3
Upon successful completion of this course, students will be able to: - explain t     3
Upon successful completion of the course, students will be able to: - understand     2
Upon successful completion of this course, students will be able to - analyze th     2
Upon successful completion of this course, students will be able to understand a     2
Upon successful completion of the course, students will be able to solve the fol     2
Upon successful completion of this course, students will be able to -define basi     1
Upon successful completion of this course, students will be able to: -understand     1
Upon successful completio


Partneři


learning_outcomes
On completion of this course, the student will be able to: A knowledge and under    18
On completion of this course, the student will be able to: Understand and discus    12
On completion of this course, the student will be able to: Understand and critic     9
1. Recognize and use appropriately important technical terms and definitions. 2.     6
On completion of this course, the student will be able to: Have had the opportun     5
On completion of this course, the student will be able to: Demonstrate a knowled     4
1. Equip with broad and useful knowledge to various topics which are not covered     4
The objectives of the course are (related to the specific BCom Learning Outcomes     3
1. Understand the logic, scientific basis, and critical thinking of economic ana     3
1. Define basic programming elements such as variables, control, loops and funct     2
On completion of this course, the student will be able to: Reflect upon and crit     2
1. Analyze firm financial

In [48]:
# odstranění placeholderů a konzervativních boilerplate prefixů z learning_outcomes

LEARNING_OUTCOMES_PLACEHOLDERS = [
    "The learning outcomes of the course has not been saved in this language version.",
    "-- item not defined --"
]

LEARNING_OUTCOMES_PATTERNS = [

    # obecné úvodní boilerplate formulace
    r"^Upon successful completion of this course, students will be able to:?\s*",
    r"^Upon successful completion of this course, student will be able to:?\s*",
    r"^Upon successful completion of the course, students will be able to:?\s*",
    r"^Upon successful completion of the course, student will be able to:?\s*",

    r"^Upon successful completion of this course, students will:?\s*",
    r"^Upon successful completion of this course, student will:?\s*",

    r"^Upon successful completion, students will be able to:?\s*",
    r"^Upon successful completion, student will be able to:?\s*",

    r"^Upon successful completion students will:?\s*",
    r"^Upon successful completion student will:?\s*",

    r"^Upon successful completion of this course,?\s*",
    r"^Upon successful completion of the course,?\s*",

    # completion / after completion
    r"^Upon completion of the course, students:?\s*",
    r"^Upon completion of the course, student:?\s*",

    r"^Upon completion of this course, students:?\s*",
    r"^Upon completion of this course, student:?\s*",

    r"^Upon completion of the course, students will:?\s*",
    r"^Upon completion of the course, student will:?\s*",

    r"^Upon completion of this course, students will:?\s*",
    r"^Upon completion of this course, student will:?\s*",

    r"^Upon the completion of the course, the students should:?\s*",
    r"^Upon the completion of the course, the student should:?\s*",

    r"^Upon the successful completion of the course,?\s*",

    r"^After successful completion of this course,?\s*",
    r"^After successful passing the course,?\s*",

    r"^After completing the course, the student:?\s*",
    r"^After completing the course, students will:?\s*",

    r"^After taking the course, the students shall be able to:?\s*",

    # by the end / on completion
    r"^By the end of the course,? the students will:?\s*",
    r"^By the end of the course,? the student will:?\s*",

    r"^By the end of this course,? the student should be able to:?\s*",

    r"^By the end of this course it is expected that the student will be able to:?\s*",

    r"^By the end of the [A-Z]{2,}\s*\d{3}\s*course it is expected that students will be able to:?\s*",

    r"^By the end of the course the candidate:?\s*",

    r"^On completion of this course, the student will be able to:?\s*",

    # objectives
    r"^The objectives of the course are \(related to the specific BCom Learning Outcomes in brackets\):?\s*",

    r"^The objectives of the course are \(related to the specific BCom Learning Outcomes\):?\s*",

    r"^The objectives of the course are:?\s*",

    r"^The broad objectives of the course are that students will demonstrate an understanding of:?\s*",

    r"^The outcomes of the course are:?\s*"
]

# administrativní instrukce k vyplnění sylabu
LEARNING_OUTCOMES_ADMIN_PATTERNS = [
    r"^All learning outcomes should be structured as bullet points:?\s*https?://\S+\s*",

    r"^The bullet points in all the learning outcome categories need an introduction\.\s*See more about structuring and wording of learning outcomes,?\s*see NOKUT:?\s*NOKUT:?\s*https?://\S+\s*",

    r"^Learning objectives should be struktured as points\.\s*Information about this here:?\s*https?://\S+\s*",

    r"^Læringsutbytte skal struktureres med kulepunkter ref\.?\s*NOKUT\s*https?://\S+\s*"
]


def clean_learning_outcomes(text):

    if pd.isna(text):
        return pd.NA

    text = str(text)

    # sjednocení whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # placeholdery
    if text in LEARNING_OUTCOMES_PLACEHOLDERS:
        return pd.NA

    # odstranění administrativních instrukcí
    for pattern in LEARNING_OUTCOMES_ADMIN_PATTERNS:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()

    # odstranění boilerplate prefixů
    for pattern in LEARNING_OUTCOMES_PATTERNS:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()

    if text == "":
        return pd.NA

    return text


# aplikace cleaningu

vse_df["learning_outcomes"] = (
    vse_df["learning_outcomes"]
    .apply(clean_learning_outcomes)
)

partner_df["learning_outcomes"] = (
    partner_df["learning_outcomes"]
    .apply(clean_learning_outcomes)
)

In [50]:
# kontrola po cleaningu learning_outcomes

print("VŠE")
print("Chybějící hodnoty:", vse_df["learning_outcomes"].isna().sum())
print("Prázdné řetězce:", (vse_df["learning_outcomes"].fillna("").str.strip() == "").sum())

display(
    vse_df["learning_outcomes"]
    .dropna()
    .str[:150]
    .value_counts()
    .head(30)
)

print("\nPartneři")
print("Chybějící hodnoty:", partner_df["learning_outcomes"].isna().sum())
print("Prázdné řetězce:", (partner_df["learning_outcomes"].fillna("").str.strip() == "").sum())

display(
    partner_df["learning_outcomes"]
    .dropna()
    .str[:150]
    .value_counts()
    .head(40)
)

VŠE
Chybějící hodnoty: 31
Prázdné řetězce: 31


learning_outcomes
- understand the content of financial statements for external users; - define and explain the nature of fundamental elements of financial statements;     3
- explain the differences in substance and management accounting system design for product costing and compare it with the accounting information for     2
- analyze theoretically economic issues - describe the relationship between economic theory and economic reality - apply basic principles of human eco    2
- explain reasons of chosen environmental problems of different environmental fields; - shortly characterize approaches of leading schools of economic    1
- work with patterns and context in real functioning socioeconomic system, including inter- action relations among nature, society and economic proces    1
- identify a problem; - propose a hypothesis; - find and critically assess available information (including academic resources in English); - formulat    1
- recognize and describe various market struct


Partneři
Chybějící hodnoty: 165
Prázdné řetězce: 165


learning_outcomes
1. Equip with broad and useful knowledge to various topics which are not covered by existing courses. 2. (Each offering under the umbrella will have s    3
1. Describe fundamental concepts and techniques for determining the asymptotic behavior of real-valued functions defined in natural numbers. 2. Explai    2
1. Identify and discuss forces that shape business activities and vice-versa. In particular, the environment and consumerism. 2. Identify and discuss     2
1. Utilize context-free grammars to identify and define the formal syntax of programming languages. 2. Apply general tools such as Bison and Flex to c    2
1. Apply appropriate modeling techniques to design software for an application of medium complexity: model and document data and processing requiremen    2
1. Develop original solutions to computer science engineering problems of significant complexity. 2. Utilize prior knowledge, independent research, pu    2
1. Demonstrate an in-depth and substantial kno

Ve sloupci `learning_outcomes` byly identifikovány systémové placeholder hodnoty `The learning outcomes of the course has not been saved in this language version.` a `-- item not defined --`, které neobsahovaly skutečný obsah sylabu. Tyto hodnoty byly převedeny na chybějící hodnoty. Současně byly odstraněny opakující se úvodní boilerplate formulace a administrativní instrukce k vyplňování výsledků učení, které nepředstavovaly obsahové informace o předmětu. Obecné formulace tvořící součást konkrétních výsledků učení byly ponechány beze změny.

## Kontrola sloupce `assessment_methods`

In [52]:
# kontrola sloupce assessment_methods

print("VŠE")
print("Chybějící hodnoty:", vse_df["assessment_methods"].isna().sum())
print("Prázdné řetězce:", (vse_df["assessment_methods"].fillna("").str.strip() == "").sum())

display(
    vse_df["assessment_methods"]
    .dropna()
    .str[:150]
    .value_counts()
    .head(30)
)

print("\nPartneři")
print("Chybějící hodnoty:", partner_df["assessment_methods"].isna().sum())
print("Prázdné řetězce:", (partner_df["assessment_methods"].fillna("").str.strip() == "").sum())

display(
    partner_df["assessment_methods"]
    .dropna()
    .str[:150]
    .value_counts()
    .head(40)
)

VŠE
Chybějící hodnoty: 0
Prázdné řetězce: 0


assessment_methods
Mid-term test(s): 40% Final test: 60%                                                                                                                     8
Mid-term test(s): 50% Final test: 50%                                                                                                                     5
Term paper: 40% Final test: 60%                                                                                                                           3
Active lecture/seminar/workshop/tutorial participation20 %: 20% Mid-term test(s)20 %: 20% Final test60 %: 60% Total100 %: 100%                            3
Active lecture/seminar/workshop/tutorial participation30 %: 0% Final test70 %: 100% Total100 %: 100%                                                      3
Active lecture/seminar/workshop/tutorial participation: 20% Final test: 80%                                                                               3
Active lecture/seminar/workshop/tutorial part


Partneři
Chybějící hodnoty: 555
Prázdné řetězce: 555


assessment_methods
Written Exam 0 %, Coursework 100 %, Practical Exam 0 %                                                                                                    35
Ordinary examination                                                                                                                                      30
Written Exam 70 %, Coursework 30 %, Practical Exam 0 %                                                                                                    13
Exam category: School Exam Form of assessment: Written School Exam - pen and paper Exam/hand-in semester: First Semester Weight: 100 Grouping: Individ    11
Exam category: School Exam Form of assessment: Written School Exam - digital Exam/hand-in semester: First Semester Weight: 100 Grouping: Individual Su    11
Written Exam 80 %, Coursework 20 %, Practical Exam 0 %                                                                                                     9
Written Exam 60 %, Coursework 40 %, Pra

In [55]:
# technické čištění assessment_methods

ASSESSMENT_METHODS_PLACEHOLDERS = [
    "-- item not defined --"
]


def clean_assessment_methods(text):

    if pd.isna(text):
        return pd.NA

    text = str(text)

    # doplnění mezery mezi textem a číslem
    text = re.sub(r"([a-zA-Z])(\d)", r"\1 \2", text)

    # doplnění mezery mezi závorkou a číslem
    text = re.sub(r"(\))(\d)", r"\1 \2", text)

    # sjednocení zápisu procent
    text = re.sub(r"(\d)\s*%", r"\1 %", text)

    # sjednocení whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # placeholdery
    if text in ASSESSMENT_METHODS_PLACEHOLDERS:
        return pd.NA

    # prázdné hodnoty
    if text == "":
        return pd.NA

    return text


# aplikace cleaningu

vse_df["assessment_methods"] = (
    vse_df["assessment_methods"]
    .apply(clean_assessment_methods)
)

partner_df["assessment_methods"] = (
    partner_df["assessment_methods"]
    .apply(clean_assessment_methods)
)

In [56]:
# kontrola po cleaningu assessment_methods

print("VŠE")
print("Chybějící hodnoty:", vse_df["assessment_methods"].isna().sum())
print("Prázdné řetězce:", (vse_df["assessment_methods"].fillna("").str.strip() == "").sum())

display(
    vse_df["assessment_methods"]
    .dropna()
    .str[:150]
    .value_counts()
    .head(30)
)

print("\nPartneři")
print("Chybějící hodnoty:", partner_df["assessment_methods"].isna().sum())
print("Prázdné řetězce:", (partner_df["assessment_methods"].fillna("").str.strip() == "").sum())

display(
    partner_df["assessment_methods"]
    .dropna()
    .str[:150]
    .value_counts()
    .head(40)
)

VŠE
Chybějící hodnoty: 0
Prázdné řetězce: 0


assessment_methods
Mid-term test(s): 40 % Final test: 60 %                                                                                                   8
Mid-term test(s): 50 % Final test: 50 %                                                                                                   5
Term paper: 40 % Final test: 60 %                                                                                                         3
Active lecture/seminar/workshop/tutorial participation 20 %: 20 % Mid-term test(s) 20 %: 20 % Final test 60 %: 60 % Total 100 %: 100 %    3
Active lecture/seminar/workshop/tutorial participation 30 %: 0 % Final test 70 %: 100 % Total 100 %: 100 %                                3
Active lecture/seminar/workshop/tutorial participation: 20 % Final test: 80 %                                                             3
Active lecture/seminar/workshop/tutorial participation: 30 % Mid-term test(s): 30 % Final test: 40 %                                      3
M


Partneři
Chybějící hodnoty: 555
Prázdné řetězce: 555


assessment_methods
Written Exam 0 %, Coursework 100 %, Practical Exam 0 %                                                                                                    35
Ordinary examination                                                                                                                                      30
Written Exam 70 %, Coursework 30 %, Practical Exam 0 %                                                                                                    13
Exam category: School Exam Form of assessment: Written School Exam - pen and paper Exam/hand-in semester: First Semester Weight: 100 Grouping: Individ    11
Exam category: School Exam Form of assessment: Written School Exam - digital Exam/hand-in semester: First Semester Weight: 100 Grouping: Individual Su    11
Written Exam 80 %, Coursework 20 %, Practical Exam 0 %                                                                                                     9
Written Exam 60 %, Coursework 40 %, Pra

Ve sloupci `assessment_methods` nebyly identifikovány žádné chybějící hodnoty u dat VŠE. U partnerských univerzit se chybějící hodnoty vyskytovaly u části záznamů, protože některé zdroje informace o způsobu hodnocení neuváděly. V rámci čištění bylo provedeno pouze technické sjednocení textu, konkrétně doplnění chybějících mezer mezi textem a číselnými hodnotami, sjednocení zápisu procent a odstranění nadbytečných mezer. Obsahové informace o způsobu hodnocení nebyly dále upravovány, protože obsahují relevantní údaje o formě zkoušky, průběžném hodnocení, váhách jednotlivých částí hodnocení nebo skupinovém či individuálním charakteru hodnocení.

## Kontrola sloupce `literature`

In [59]:
# kontrola sloupce literature

print("VŠE")
print("Chybějící hodnoty:", vse_df["literature"].isna().sum())
print("Prázdné řetězce:", (vse_df["literature"].fillna("").str.strip() == "").sum())

display(
    vse_df["literature"]
    .dropna()
    .str[:150]
    .value_counts()
    .head(30)
)

print("\nPartneři")
print("Chybějící hodnoty:", partner_df["literature"].isna().sum())
print("Prázdné řetězce:", (partner_df["literature"].fillna("").str.strip() == "").sum())

display(
    partner_df["literature"]
    .dropna()
    .str[:150]
    .value_counts()
    .head(40)
)

VŠE
Chybějící hodnoty: 0
Prázdné řetězce: 0


literature
Basic: GÉRON, Aurélien. Hands-on machine learning with Scikit-Learn, Keras and TensorFlow : concepts, tools, and techniques to build intelligent syste    2
Basic: Votava, T. et al. (2024). Základy obchodního práva po rekodifikaci soukromého práva (6th ed.). Vydavatelství a nakladatelství Aleš Čenek, s.r.o    2
Basic: Grmelová, N., Andreisová, L., & Vaňkátová, T. (2019). An Introduction to Law (1st ed.). Powerprint, s.r.o. Recommended: CRAIG, Paul P.; DE BÚRC    2
Basic: Chiang, A. C., & Wainwright, K. (2005). Fundamental methods of mathematical economics. McGraw-Hill. Klůfa, J., & Kaspříková, N. (2013). Mathema    1
Basic: CIMLER, Petr; ZADRAŽILOVÁ, Dana. Retail management. Praha: Management Press, 2007. ISBN 978-80-7261-167-6.                                         1
Basic: Kubálek, T., & Kubálková, M. (2015). Program pro řízení projektů Microsoft Project 2013 (1st ed.). Nakladatelství Oeconomica. Kubálek, T., & Ku    1
Basic: Kubálek, T., & Kubálková, M. (2016). Databázov


Partneři
Chybějící hodnoty: 692
Prázdné řetězce: 692


literature
Textbooks / Resources There is no required text book for the course.                                                                                      2
Readings may be found either in the University library collection or accessed via The Library electronic journals on http://www.ed.ac.uk/schools-depar    2
Textbooks / Resources This course has weekly online readings for students which can be access via the library and uploaded onto Learn where possible.     2
Students will find the reading list on Learn.                                                                                                             2
Textbooks / Resources There is no prescribed textbook for the course, as the lecture notes, videos and learning modules should be sufficient for most     2
Mankiw and Taylor 'Economics'                                                                                                                             2
Textbooks / Resources Required Texts Shirley Carlon; 

In [65]:
# čištění literature

LITERATURE_PLACEHOLDERS = [
    "-- item not defined --"
]

LITERATURE_ADMIN_PATTERNS = [
    r"^Students will find the reading list on Learn\.?$",

    r"^This course has weekly online readings for students.*$",

    r"^Readings may be found either in the University library collection.*$",

    r"^There is no required text book for the course\.?$",

    r"^There is no prescribed textbook for the course.*$",

    r"^A resource list will be published in Week 1\..*$",

    r"^A programme of study with topics and selected readings will be provided at the start of the course\.?$"
]


def clean_literature(text):

    if pd.isna(text):
        return pd.NA

    text = str(text)

    # sjednocení whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # odstranění boilerplate prefixu
    text = re.sub(
        r"^Textbooks\s*/\s*Resources\s*",
        "",
        text,
        flags=re.IGNORECASE
    ).strip()

    # placeholdery
    if text in LITERATURE_PLACEHOLDERS:
        return pd.NA

    # administrativní / neobsahové informace
    for pattern in LITERATURE_ADMIN_PATTERNS:
        if re.match(pattern, text, flags=re.IGNORECASE):
            return pd.NA

    # čisté URL bez dalšího textu
    if re.fullmatch(r"https?://\S+", text):
        return pd.NA

    # prázdné hodnoty
    if text == "":
        return pd.NA

    return text


vse_df["literature"] = (
    vse_df["literature"]
    .apply(clean_literature)
)

partner_df["literature"] = (
    partner_df["literature"]
    .apply(clean_literature)
)

In [66]:
# kontrola po cleaningu literature

print("VŠE")
print("Chybějící hodnoty:", vse_df["literature"].isna().sum())
print("Prázdné řetězce:", (vse_df["literature"].fillna("").str.strip() == "").sum())

display(
    vse_df["literature"]
    .dropna()
    .str[:150]
    .value_counts()
    .head(30)
)

print("\nPartneři")
print("Chybějící hodnoty:", partner_df["literature"].isna().sum())
print("Prázdné řetězce:", (partner_df["literature"].fillna("").str.strip() == "").sum())

display(
    partner_df["literature"]
    .dropna()
    .str[:150]
    .value_counts()
    .head(40)
)

VŠE
Chybějící hodnoty: 0
Prázdné řetězce: 0


literature
Basic: GÉRON, Aurélien. Hands-on machine learning with Scikit-Learn, Keras and TensorFlow : concepts, tools, and techniques to build intelligent syste    2
Basic: Grmelová, N., Andreisová, L., & Vaňkátová, T. (2019). An Introduction to Law (1st ed.). Powerprint, s.r.o. Recommended: CRAIG, Paul P.; DE BÚRC    2
Basic: Votava, T. et al. (2024). Základy obchodního práva po rekodifikaci soukromého práva (6th ed.). Vydavatelství a nakladatelství Aleš Čenek, s.r.o    2
Basic: Holman, R. (2005). Ekonomie. C.H. Beck. Recommended: Macáková, L. (2010). Mikroekonomie : základní kurs. Melandrium. Pavelka, T. (2007). Makroe    1
Basic: Mishkin, F. S. (2016). The economics of money, banking and financial markets. Pearson. Tresch, R. W. (2008). Public sector economics. Palgrave     1
Basic: Revenda, Z., Mandel, M., Kodera, J., Musílek, P., & Dvořák, P. (2023). Peněžní ekonomie a bankovnictví (7th ed.). Ekopress, s.r.o. Recommended:    1
Basic: Radová, J., Dvořák, P., & Málek, J. (2013). Fi


Partneři
Chybějící hodnoty: 708
Prázdné řetězce: 708


literature
Mankiw and Taylor 'Economics'                                                                                                                             2
Required Texts Atrill, Peter; ACCT102 accounting and financial infomation ; 4th edition; Pearson.                                                         1
The course follows the textbook: D. Besanko, D. Dranove, M. Shanley, S. Schaefer, Economics of Strategy, 6th Edition, Wiley, 2013. (Students are stron    1
There are many good text books available on organisation change and change management. The core text for the course is: Hughes, Mark (2010). Managing     1
Students will require to consult an up to date text book on employment law. The recommended text is: - Stephen Taylor and Astra Emir (2019) (5th Editi    1
J Rees & D Mullins (2016) The third sector delivering public services (Bristol University Press) M Kallman & T Clark (2016) The Third Sector (Universi    1
Recommended Reading: 1. S. P. Bradley, A. C. Hax, and

Ve sloupci `literature` nebyly u dat VŠE identifikovány chybějící hodnoty. U partnerských univerzit se chybějící hodnoty vyskytovaly častěji, protože některé instituce literaturu vůbec neuváděly nebo ji zpřístupňovaly prostřednictvím externích systémů. V rámci čištění bylo provedeno sjednocení formátování textu, odstranění placeholder hodnot, samostatných URL odkazů bez další obsahové informace a vybraných administrativních formulací neobsahujících konkrétní bibliografické údaje. Samostatné URL odkazy byly odstraněny, protože bez doprovodného textu nepředstavovaly využitelnou informaci pro následné porovnávání sylabů. Bibliografické záznamy, doporučená literatura, odkazy uvedené jako součást textu a další obsahové informace byly ponechány beze změny.

## Kontrola obsahových atributů

In [69]:
# kontrola záznamů bez všech tří obsahových atributů podle univerzity

content_columns = ["aims", "course_content", "learning_outcomes"]

def all_content_missing(df):
    return df[content_columns].isna().all(axis=1)

print("VŠE")

display(
    vse_df.loc[
        all_content_missing(vse_df)
    ]
    .groupby("university")
    .size()
    .reset_index(name="missing_content_records")
)

print("\nPartneři")

display(
    partner_df.loc[
        all_content_missing(partner_df)
    ]
    .groupby("university")
    .size()
    .reset_index(name="missing_content_records")
)

VŠE


,university,missing_content_records
0,Prague University of Economics and Business,30



Partneři


,university,missing_content_records
0,University of Canterbury,1


In [74]:
# odstranění záznamů bez dostupného obsahového atributu

vse_before = len(vse_df)
partner_before = len(partner_df)

vse_df = vse_df.loc[~all_content_missing(vse_df)].copy()
partner_df = partner_df.loc[~all_content_missing(partner_df)].copy()

print("Odstraněno z VŠE:", vse_before - len(vse_df))
print("Odstraněno z partnerských univerzit:", partner_before - len(partner_df))
print("Počet záznamů VŠE po odstranění:", len(vse_df))
print("Počet záznamů partnerů po odstranění:", len(partner_df))

Odstraněno z VŠE: 30
Odstraněno z partnerských univerzit: 1
Počet záznamů VŠE po odstranění: 103
Počet záznamů partnerů po odstranění: 898


Byly identifikovány a odstraněny záznamy, u nichž nebyl dostupný ani jeden z hlavních obsahových atributů (`aims`, `course_content`, `learning_outcomes`). Jednalo se především o část anglických verzí sylabů VŠE, ve kterých nebyly obsahové informace zveřejněny. Dataset tak obsahuje pouze předměty s alespoň jedním dostupným obsahovým atributem využitelným pro následné porovnávání sylabů.

## Finální statistika datasetu

In [77]:
# základní statistika datasetu - VŠE

print("VŠE")
print("Počet záznamů:", len(vse_df))
print("Počet sloupců:", vse_df.shape[1])

print("\nPočet záznamů podle akademického roku:")
display(
    vse_df["academic_year"]
    .fillna("missing")
    .value_counts()
    .rename_axis("academic_year")
    .reset_index(name="count")
)

print("\nDostupnost hlavních textových atributů:")
display(
    vse_df[
        [
            "aims",
            "course_content",
            "learning_outcomes",
            "assessment_methods",
            "literature"
        ]
    ]
    .notna()
    .sum()
    .rename("non_missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

VŠE
Počet záznamů: 103
Počet sloupců: 17

Počet záznamů podle akademického roku:


,academic_year,count
0,2026/2027,75
1,2025/2026,28



Dostupnost hlavních textových atributů:


,column,non_missing_count
0,aims,103
1,course_content,102
2,learning_outcomes,102
3,assessment_methods,103
4,literature,103


In [80]:
# základní statistika datasetu - partnerské univerzity

print("Partneři")
print("Počet záznamů:", len(partner_df))
print("Počet sloupců:", partner_df.shape[1])

print("\nPočet záznamů podle univerzity:")
display(
    partner_df["university"]
    .value_counts()
    .rename_axis("university")
    .reset_index(name="count")
)

print("\nPočet záznamů podle země:")
display(
    partner_df["country"]
    .value_counts()
    .rename_axis("country")
    .reset_index(name="count")
)

print("\nPočet záznamů podle úrovně studia:")
display(
    partner_df["study_level"]
    .fillna("missing")
    .value_counts()
    .rename_axis("study_level")
    .reset_index(name="count")
)

print("\nPočet záznamů podle akademického roku:")
display(
    partner_df["academic_year"]
    .fillna("missing")
    .value_counts()
    .rename_axis("academic_year")
    .reset_index(name="count")
)

print("\nDostupnost hlavních textových atributů:")
text_columns = [
    "aims",
    "course_content",
    "learning_outcomes",
    "assessment_methods",
    "literature"
]

availability = (
    partner_df[text_columns]
    .notna()
    .sum()
    .rename("non_missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

availability["availability_percent"] = (
    availability["non_missing_count"] / len(partner_df) * 100
).round(1)

display(availability)

Partneři
Počet záznamů: 898
Počet sloupců: 17

Počet záznamů podle univerzity:


,university,count
0,The Hong Kong University of Science and Technology,382
1,University of Canterbury,188
2,BI Norwegian Business School,126
3,University of Edinburgh,98
4,Queen's University,71
5,Instituto Tecnológico Autónomo de México,33



Počet záznamů podle země:


,country,count
0,Hong Kong,382
1,New Zealand,188
2,Norway,126
3,United Kingdom,98
4,Canada,71
5,Mexico,33



Počet záznamů podle úrovně studia:


,study_level,count
0,undergraduate,772
1,master,71
2,bachelor,55



Počet záznamů podle akademického roku:


,academic_year,count
0,2025/2026,688
1,2026/2027,124
2,2024/2025,75
3,2016/2017,3
4,2023/2024,3
5,2019/2020,2
6,2018/2019,1
7,2015/2016,1
8,2019,1



Dostupnost hlavních textových atributů:


,column,non_missing_count,availability_percent
0,aims,897,99.9
1,course_content,254,28.3
2,learning_outcomes,734,81.7
3,assessment_methods,344,38.3
4,literature,191,21.3


## Export finálních datasetů

In [79]:
# export finálních datasetů

vse_output_path = PROCESSED_DIR / "vse_syllabi_processed.csv"
partner_output_path = PROCESSED_DIR / "partner_syllabi_processed.csv"

vse_df.to_csv(vse_output_path, index=False, encoding="utf-8-sig")
partner_df.to_csv(partner_output_path, index=False, encoding="utf-8-sig")

print(f"VŠE dataset uložen do: {vse_output_path}")
print(f"Partnerský dataset uložen do: {partner_output_path}")

VŠE dataset uložen do: C:\Users\semlu\Documents\syllabus-similarity-thesis\data\processed\vse_syllabi_processed.csv
Partnerský dataset uložen do: C:\Users\semlu\Documents\syllabus-similarity-thesis\data\processed\partner_syllabi_processed.csv
